# Yambda 03 · Baselines

Same evaluator as Last.fm (`src/musicrec/evaluation.py`). Two target sets:

* **main** (`split.targets`): organic + valid listen + never heard in train
* **all** (`targets_all.parquet`): valid listen + never heard, organic *or* recommended (secondary)

Validation split only. ~30 s per model.

In [1]:
import pandas as pd
from musicrec import config
from musicrec.preprocess import Split
from musicrec.evaluation import evaluate
from musicrec.models.baselines import Random, MostPopular

pd.set_option("display.float_format", "{:,.4f}".format)
split = Split.load(config.YAMBDA_SPLIT_VAL_DIR)
targets_all = pd.read_parquet(config.YAMBDA_SPLIT_VAL_DIR / "targets_all.parquet")
print(f"{split.n_users:,} users x {split.n_items:,} items | main targets {len(split.targets):,} | all targets {len(targets_all):,}")

8,593 users x 160,613 items | main targets 107,165 | all targets 378,635


In [2]:
COLS = ["recall@10", "ndcg@10", "hit_rate@10", "recall@50", "coverage@10", "avg_pop@10", "n_eval_users"]
models = [Random(), MostPopular(by="plays"), MostPopular(by="listeners"),
          MostPopular(by="listeners", recent_days=30), MostPopular(by="listeners", recent_days=14),
          MostPopular(by="listeners", recent_days=7)]
main, secondary = {}, {}
for m in models:
    m.fit(split)
    main[m.name] = evaluate(m, split)
    secondary[m.name] = evaluate(m, split, targets=targets_all)
    print(f"{m.name:<40} main recall@10={main[m.name]['recall@10']:.4f}")
baselines = pd.DataFrame(main).T[COLS]
baselines

Random                                   main recall@10=0.0004
MostPopular(plays)                       main recall@10=0.0127
MostPopular(listeners)                   main recall@10=0.0121
MostPopular(listeners, last 30d)         main recall@10=0.0211
MostPopular(listeners, last 14d)         main recall@10=0.0225
MostPopular(listeners, last 7d)          main recall@10=0.0235


,recall@10,ndcg@10,hit_rate@10,recall@50,coverage@10,avg_pop@10,n_eval_users
Random,0.0004,0.0004,0.0026,0.0005,0.2974,43.9013,"5,678.0000"
MostPopular(plays),0.0127,0.0121,0.0667,0.0202,0.0009,"2,708.5291","5,678.0000"
MostPopular(listeners),0.0121,0.0115,0.0652,0.0194,0.0009,"2,807.2969","5,678.0000"
"MostPopular(listeners, last 30d)",0.0211,0.0192,0.0946,0.0350,0.0008,"2,250.6270","5,678.0000"
"MostPopular(listeners, last 14d)",0.0225,0.0211,0.1011,0.0379,0.0008,"2,041.1047","5,678.0000"
"MostPopular(listeners, last 7d)",0.0235,0.0224,0.1062,0.0428,0.0007,"1,819.4269","5,678.0000"


In [3]:
# Secondary targets (organic + recommended discoveries)
pd.DataFrame(secondary).T[["recall@10", "ndcg@10", "hit_rate@10", "n_eval_users"]]

,recall@10,ndcg@10,hit_rate@10,n_eval_users
Random,0.0003,0.0003,0.0032,"7,145.0000"
MostPopular(plays),0.0181,0.0183,0.1059,"7,145.0000"
MostPopular(listeners),0.0190,0.0197,0.1131,"7,145.0000"
"MostPopular(listeners, last 30d)",0.0338,0.0335,0.1794,"7,145.0000"
"MostPopular(listeners, last 14d)",0.0363,0.0358,0.1895,"7,145.0000"
"MostPopular(listeners, last 7d)",0.0391,0.0401,0.2036,"7,145.0000"


In [4]:
out = config.PROJECT_ROOT / "reports" / "yambda"
out.mkdir(parents=True, exist_ok=True)
baselines.to_csv(out / "results_val_baselines.csv")